In [ ]:
import pandas as pd
import networkx as nx
from pathlib import Path


# 1. Function to calculate centralities


def calculate_centrality(file, antibiotic):

    edges = pd.read_csv(file)

    # Directed GRN: regulator → target
    G = nx.from_pandas_edgelist(
        edges,
        source=edges.columns[0],
        target=edges.columns[1],
        create_using=nx.DiGraph()
    )

    # Degree: downstream targets
    degree = dict(G.out_degree())

    # Betweenness
    betweenness = nx.betweenness_centrality(
        G,
        normalized=True
    )

    # Closeness: downstream direction
    closeness = nx.closeness_centrality(
        G.reverse()
    )

    centrality = pd.DataFrame({
        "gene": list(G.nodes()),
        "degree": [degree[g] for g in G.nodes()],
        "betweenness": [betweenness[g] for g in G.nodes()],
        "closeness": [closeness[g] for g in G.nodes()]
    })

    centrality["antibiotic"] = antibiotic

    return centrality

In [2]:
networks = {
    "Amikacin": "ami.grn.csv",
    "Ciprofloxacin": "cip.grn.csv",
    "Meropenem": "mero.grn.csv",
    "Polymyxin": "poly.grn.csv",
    "Doripenem": "dori.grn.csv",
    "Tigecycline": "tige.grn.csv",
    "Colistin-resistant": "colis.resis.grn.csv"
}

In [3]:
all_centrality = []

for antibiotic, file in networks.items():

    result = calculate_centrality(
        file,
        antibiotic
    )

    all_centrality.append(result)


all_centrality = pd.concat(
    all_centrality,
    ignore_index=True
)

In [4]:
top10_results = []

for antibiotic, group in all_centrality.groupby("antibiotic"):

    # Top 10 Degree
    degree_top = (
        group.sort_values("degree", ascending=False)
        .head(10)
        .copy()
    )
    degree_top["measure"] = "Degree"
    degree_top["rank"] = range(1, 11)

    # Top 10 Betweenness
    betweenness_top = (
        group.sort_values("betweenness", ascending=False)
        .head(10)
        .copy()
    )
    betweenness_top["measure"] = "Betweenness"
    betweenness_top["rank"] = range(1, 11)

    # Top 10 Closeness
    closeness_top = (
        group.sort_values("closeness", ascending=False)
        .head(10)
        .copy()
    )
    closeness_top["measure"] = "Closeness"
    closeness_top["rank"] = range(1, 11)

    top10_results.extend([
        degree_top,
        betweenness_top,
        closeness_top
    ])

In [5]:
top10_all = pd.concat(
    top10_results,
    ignore_index=True
)

In [6]:
top10_all.to_csv(
    "Top10_all_antibiotics_centralities.csv",
    index=False
)

In [7]:
common_results = []

for antibiotic, group in all_centrality.groupby("antibiotic"):

    top_degree = set(
        group.nlargest(10, "degree")["gene"]
    )

    top_betweenness = set(
        group.nlargest(10, "betweenness")["gene"]
    )

    top_closeness = set(
        group.nlargest(10, "closeness")["gene"]
    )

    common = (
        top_degree
        & top_betweenness
        & top_closeness
    )

    for gene in common:

        row = group[group["gene"] == gene].iloc[0]

        common_results.append({
            "antibiotic": antibiotic,
            "gene": gene,
            "degree": row["degree"],
            "betweenness": row["betweenness"],
            "closeness": row["closeness"]
        })


common_df = pd.DataFrame(common_results)

common_df.to_csv(
    "common_top10_three_centralities.csv",
    index=False
)